# Bill Total & Date Extractor v3 – higher accuracy (English + Sinhala/Tamil)

**Why v1 was only ~80% on totals**, and what this version changes:

| Problem in v1 | Fix in v3 |
|---|---|
| Trained on only **75** receipts | Trains on up to **all 626** SROIE receipts (+ your own bills), OCR cached to Drive so it is only done once |
| Weak features (position, rank, keywords) | +14 features: arithmetic hints (*total = subtotal + tax*, *total + change = cash*), 5-cent rounding, position order, row layout, next-line keywords, fuzzy keywords (`T0TAL`) |
| One model, never compared | Random Forest vs Extra Trees vs Gradient Boosting vs a voting ensemble, **chosen by grouped cross-validation** |
| One OCR pass | Two OCR "views" (grey / thresholded); a number that scores high in both wins |
| Date = first date found | Prefers dates written after the word *date*, then the most repeated, with spaced-separator support |
| No error analysis | Every failure is labelled **OCR miss** (digits not read) or **ranking miss** (read but wrong one picked) |

Run cells in order. Cell 5 (own bills) is optional but gives the biggest gain on Sri Lankan bills.

## Cell 1 – Install, best OCR models, Drive cache

In [1]:
!apt-get -qq install -y tesseract-ocr tesseract-ocr-sin tesseract-ocr-tam > /dev/null
!pip install -q pytesseract python-dateutil opencv-python-headless rapidfuzz

!mkdir -p /content/tessdata_best
!for L in eng sin tam; do wget -q -O /content/tessdata_best/$L.traineddata https://github.com/tesseract-ocr/tessdata_best/raw/main/$L.traineddata; done
!tesseract --list-langs --tessdata-dir /content/tessdata_best

# Cache OCR results on Drive so a runtime reset does not cost you the slow step
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = "/content/drive/MyDrive/bill_extractor_cache"
except Exception:
    CACHE_DIR = "/content/bill_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
print("OCR cache folder:", CACHE_DIR)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 48.2 MB/s eta 0:00:00
List of available languages in "/content/tessdata_best/" (3):
eng
sin
tam
Mounted at /content/drive
OCR cache folder: /content/drive/MyDrive/bill_extractor_cache


## Cell 2 – Load SROIE receipts

In [2]:
from datasets import load_dataset
ds = load_dataset("vishu12121/ICDAR2019-SROIE")
train_ds, test_ds = ds["train"], ds["test"]
print(len(train_ds), "train receipts,", len(test_ds), "test receipts")

README.md:   0%|          | 0.00/14.3k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  319MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  191MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/626 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/361 [00:00<?, ? examples/s]

626 train receipts, 361 test receipts


## Cell 3 – Core library (preprocessing, OCR, features, dates, prediction)

In [3]:
import os, re, json, hashlib, pickle
os.environ.setdefault("OMP_THREAD_LIMIT", "1")      # 1 thread per Tesseract call -> faster with many workers
from collections import Counter
import cv2
import numpy as np
import pandas as pd
import pytesseract
from PIL import Image, ImageOps
from dateutil import parser as dparser
from rapidfuzz import fuzz

TESSDIR = "/content/tessdata_best"

# ------------------------------------------------------------------ keywords
KW = {
    "kw_total": ["total", "amount due", "amount payable", "payable", "nett", "net amount", "grand",
                 "මුළු", "මුළු මුදල", "මුළු එකතුව", "එකතුව", "ගෙවිය යුතු", "ගෙවිය යුතු මුදල",
                 "மொத்த", "மொத்தம்", "மொத்த தொகை"],
    "kw_sub": ["sub total", "subtotal", "sub-total", "උප එකතුව"],
    "kw_cash": ["cash", "tender", "paid", "change", "balance", "refund", "card", "visa", "master",
                "මුදල්", "ගෙවූ", "ඉතිරි"],
    "kw_tax": ["tax", "gst", "vat", "sst", "service", "බදු"],
    "kw_disc": ["disc", "saving", "වට්ටම්"],
    "kw_round": ["round"],
    "kw_item": ["item", "qty", "quantity", "ප්‍රමාණය"],
}
BASE = ["rel_y", "rel_x", "rank", "is_max", "n_same", "n_cands",
        "log_v", "ratio_max", "rank_y", "row_n", "rightmost", "is_sum_pair", "is_minus_pair", "mod5"]
FEATURES = BASE + list(KW) + [k + "_prev" for k in KW] + [k + "_next" for k in KW]

def kw_hit(row, kws):
    """Exact match, plus fuzzy match so OCR errors like 'T0TAL' / garbled Sinhala still hit."""
    toks = None
    for s in kws:
        if s in row:
            return True
        if not s.isascii():
            if len(row) >= 3 and fuzz.partial_ratio(s, row) >= 80:
                return True
        elif " " not in s and len(s) >= 5:
            if toks is None:
                toks = re.findall(r"[^\s:;|]+", row)
            if any(abs(len(t) - len(s)) <= 1 and fuzz.ratio(t, s) >= 80 for t in toks):
                return True
    return False

# ------------------------------------------------------------------ parsing
MONTHS = "jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec"
AMOUNT_RE = re.compile(r"\d{1,3}(?:,\d{3})+(?:\.\d{2})?|\d+[.,]\d{2}(?!\d)")
DATE_RE = re.compile(                       # used on single OCR tokens (to keep dates out of amounts)
    r"\d{4}[/.-]\d{1,2}[/.-]\d{1,2}|\d{1,2}[/.-]\d{1,2}[/.-]\d{2,4}"
    r"|\d{1,2}\s?(?:%s)[a-z]*\s?\d{2,4}" % MONTHS, re.I)
DATE_TEXT_RE = re.compile(                  # used on whole-receipt text (tolerates spaces around separators)
    r"(?<!\d)(?:\d{4}\s?[/.\-]\s?\d{1,2}\s?[/.\-]\s?\d{1,2}"
    r"|\d{1,2}\s?[/.\-]\s?\d{1,2}\s?[/.\-]\s?\d{2,4}"
    r"|\d{1,2}[\s\-]?(?:%s)[a-z]*[\s\-,]*\d{2,4})(?!\d)" % MONTHS, re.I)

def to_amount(s):
    if s is None:
        return None
    s = re.sub(r"[^\d.,]", "", str(s)).strip(".,")
    if not s:
        return None
    if "," in s and "." in s:
        s = s.replace(",", "")
    elif "," in s:
        s = s.replace(",", "") if len(s.split(",")[-1]) == 3 else s.replace(",", ".")
    try:
        return round(float(s), 2)
    except ValueError:
        return None

def to_date(s):
    if not s:
        return None
    s = str(s)
    yf = bool(re.match(r"\d{4}([./-]|\d{4}$)", s.strip()))
    try:
        return dparser.parse(s, dayfirst=not yf, yearfirst=yf, fuzzy=True).date().isoformat()
    except (ValueError, OverflowError):
        return None

def find_date(text):
    """All dates in the text -> prefer one right after the word 'date', then the most repeated, then the earliest."""
    low = text.lower()
    hits = []
    for m in DATE_TEXT_RE.finditer(text):
        raw = re.sub(r"\s*([/.\-])\s*", r"\1", m.group(0))
        raw = re.sub(r"(?<=\d)(?=[A-Za-z])|(?<=[A-Za-z])(?=\d)", " ", raw)
        d = to_date(raw)
        if d and 2000 <= int(d[:4]) <= 2035:
            hits.append((d, m.start(), "date" in low[max(0, m.start() - 25):m.start()]))
    if not hits:
        return None
    cnt = Counter(h[0] for h in hits)
    return max(hits, key=lambda h: (h[2], cnt[h[0]], -h[1]))[0]

def vote_date(texts):
    ds = [d for d in (find_date(t) for t in texts) if d]
    if not ds:
        return None
    c = Counter(ds)
    return max(ds, key=lambda d: (c[d], -ds.index(d)))

def find_currency(text):
    t = text.lower()
    if re.search(r"(?<![a-z])rs(?![a-z])|lkr|රු|ரூ", t):
        return "LKR"
    if "usd" in t or "$" in t:
        return "USD"
    if re.search(r"(?<![a-z])rm(?![a-z])", t):
        return "MYR"
    return None

# ------------------------------------------------------------------ preprocessing + OCR
def deskew(gray, max_angle=5, step=0.5):
    small = cv2.resize(gray, None, fx=0.5, fy=0.5) if gray.shape[1] > 1000 else gray
    inv = cv2.bitwise_not(cv2.threshold(small, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1])
    h, w = inv.shape
    best, best_score = 0.0, -1.0
    for a in np.arange(-max_angle, max_angle + step, step):
        M = cv2.getRotationMatrix2D((w / 2, h / 2), a, 1)
        r = cv2.warpAffine(inv, M, (w, h), flags=cv2.INTER_NEAREST)
        score = np.var(r.sum(axis=1))
        if score > best_score:
            best, best_score = a, score
    H, W = gray.shape
    M = cv2.getRotationMatrix2D((W / 2, H / 2), best, 1)
    return cv2.warpAffine(gray, M, (W, H), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

def preprocess(img, binarize=False, target_w=1600, do_deskew=True):
    g = np.array(ImageOps.exif_transpose(img).convert("L"))
    h, w = g.shape
    g = cv2.resize(g, (target_w, int(h * target_w / w)),
                   interpolation=cv2.INTER_CUBIC if w < target_w else cv2.INTER_AREA)
    g = cv2.medianBlur(g, 3)
    if do_deskew:
        g = deskew(g)
    if binarize:
        g = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15)
    return Image.fromarray(g)

def run_tess(img, lang, psm=6):
    cfg = f"--psm {psm}"
    if os.path.isdir(TESSDIR):
        cfg += f" --tessdata-dir {TESSDIR}"
    d = pytesseract.image_to_data(img, lang=lang, config=cfg, output_type=pytesseract.Output.DICT)
    return [{"t": t.strip(), "x": x, "w": w, "yc": y + h / 2, "h": h}
            for t, x, y, w, h in zip(d["text"], d["left"], d["top"], d["width"], d["height"]) if t.strip()]

def ocr_words(img, binarize=False, target_w=1600, psm=6, extra_langs=()):
    """English pass -> digits and English words. Extra-language passes (sin/tam) add non-numeric words only."""
    g = preprocess(img, binarize, target_w)
    words = run_tess(g, "eng", psm)
    for lang in extra_langs:
        words += [w for w in run_tess(g, lang, psm) if not any(c.isdigit() for c in w["t"])]
    return words, g.height

def words_text(words):
    return " ".join(w["t"] for w in sorted(words, key=lambda w: (round(w["yc"] / 10), w["x"])))

def row_text(words, yc, h, shift=0.0):
    target = yc + shift * h
    row = [w for w in words if abs(w["yc"] - target) < 0.6 * max(h, w["h"])]
    return " ".join(w["t"] for w in sorted(row, key=lambda w: w["x"])).lower()

# ------------------------------------------------------------------ candidates + features
def candidates(words, height):
    cands = []
    for w in words:
        if DATE_RE.search(w["t"]):
            continue
        for m in AMOUNT_RE.findall(w["t"]):
            v = to_amount(m)
            if v and v > 0:
                cands.append({"value": v, "yc": w["yc"], "h": w["h"], "right": w["x"] + w["w"]})
    if not cands:
        return pd.DataFrame()
    vals = np.array([c["value"] for c in cands])
    ys = np.array([c["yc"] for c in cands])
    cents = np.round(vals * 100).astype(np.int64)
    cset = set(cents.tolist())
    width = max(w["x"] + w["w"] for w in words)
    vmax = vals.max()
    rows = []
    for c, cv in zip(cands, cents.tolist()):
        same = row_text(words, c["yc"], c["h"])
        prev = row_text(words, c["yc"], c["h"], shift=-1.8)
        nxt = row_text(words, c["yc"], c["h"], shift=1.8)
        in_row = [o for o in cands if abs(o["yc"] - c["yc"]) < 0.6 * max(c["h"], o["h"])]
        f = {"value": c["value"], "rel_y": c["yc"] / height, "rel_x": c["right"] / width,
             "rank": (vals > c["value"]).sum() / len(vals), "is_max": float(c["value"] == vmax),
             "n_same": float((vals == c["value"]).sum()), "n_cands": float(len(vals)),
             "log_v": float(np.log1p(c["value"])), "ratio_max": float(c["value"] / vmax),
             "rank_y": float((ys < c["yc"]).sum() / len(ys)),
             "row_n": float(len(in_row)),
             "rightmost": float(c["right"] >= max(o["right"] for o in in_row)),
             # arithmetic hints: total = a + b (subtotal + tax)  |  total + change = cash
             "is_sum_pair": float(any((cv - a) in cset for a in cset if 0 < a < cv)),
             "is_minus_pair": float(any((cv + a) in cset for a in cset)),
             "mod5": float(cv % 5 == 0)}
        for k, kws in KW.items():
            f[k] = float(kw_hit(same, kws))
            f[k + "_prev"] = float(kw_hit(prev, kws))
            f[k + "_next"] = float(kw_hit(nxt, kws))
        rows.append(f)
    return pd.DataFrame(rows)

# ------------------------------------------------------------------ document-level prediction
def total_scores(views, clf):
    """views = [(words, height), ...]. A value that scores well in several OCR views gets a higher total score."""
    agg = {}
    for words, height in views:
        df = candidates(words, height)
        if df.empty:
            continue
        p = clf.predict_proba(df[FEATURES])[:, 1]
        best = {}
        for v, pp in zip(df["value"], p):
            best[v] = max(best.get(v, 0.0), float(pp))
        for v, pp in best.items():
            agg[v] = agg.get(v, 0.0) + pp
    return agg

def predict_views(views, clf):
    agg = total_scores(views, clf)
    total = max(agg, key=agg.get) if agg else None
    texts = [words_text(w) for w, _ in views]
    return {"total": total, "date": vote_date(texts), "currency": find_currency(" ".join(texts))}

## Cell 4 – Pick the best OCR settings (about 3 minutes)
Tries four preprocessing settings on 30 receipts. **OCR ceiling** is the most important number: if the total is not read by OCR, no model can find it. Set `VIEWS` below to the best two settings.

In [4]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

WORKERS = max(2, os.cpu_count() or 2)
CONFIGS = {
    "gray_1600_psm6": dict(binarize=False, target_w=1600, psm=6),
    "gray_2000_psm6": dict(binarize=False, target_w=2000, psm=6),
    "bin_2000_psm6":  dict(binarize=True,  target_w=2000, psm=6),
    "gray_1600_psm4": dict(binarize=False, target_w=1600, psm=4),
}
N_CHECK = 30
def check(i, cfg):
    ex = train_ds[i]
    w, h = ocr_words(ex["image"], **cfg)
    tot = to_amount(ex["entities"]["total"]); dt = to_date(ex["entities"]["date"])
    c = candidates(w, h)
    return (tot is not None and not c.empty and bool(((c["value"] - tot).abs() < 0.005).any()),
            find_date(words_text(w)) == dt)

rows = []
for name, cfg in CONFIGS.items():
    with ThreadPoolExecutor(WORKERS) as pool:
        r = list(pool.map(lambda i: check(i, cfg), range(N_CHECK)))
    rows.append({"config": name, "OCR ceiling %": 100 * np.mean([a for a, _ in r]),
                 "date found %": 100 * np.mean([b for _, b in r])})
display(pd.DataFrame(rows).round(1))

,config,OCR ceiling %,date found %
0,gray_1600_psm6,76.7,76.7
1,gray_2000_psm6,80.0,80.0
2,bin_2000_psm6,76.7,73.3
3,gray_1600_psm4,83.3,83.3


In [5]:
# >>> Set the two best configs from the table above. Default = grey + thresholded.
VIEWS = [
    dict(binarize=False, target_w=1600, psm=6),
    dict(binarize=True,  target_w=2000, psm=6),
]
SIG = hashlib.md5(json.dumps(VIEWS, sort_keys=True).encode()).hexdigest()[:8]
print("views:", VIEWS, "| cache signature:", SIG)

views: [{'binarize': False, 'target_w': 1600, 'psm': 6}, {'binarize': True, 'target_w': 2000, 'psm': 6}] | cache signature: 25ad0a14


## Cell 5 – OCR the receipts (slow the first time, instant afterwards)
`N_TRAIN` / `N_TEST` default to **all** receipts. Lower them (e.g. 200 / 80) for a quick first run. Results are cached to Drive in chunks, so if Colab disconnects, just run this cell again and it continues.

In [6]:
N_TRAIN = len(train_ds)      # 626 -> all
N_TEST  = 150                # 361 available; 150 is a solid test set

def ocr_docs(dset, n, split, extra_langs=()):
    path = f"{CACHE_DIR}/{split}_{SIG}.pkl"
    cache = pickle.load(open(path, "rb")) if os.path.exists(path) else {}
    todo = [i for i in range(n) if i not in cache]
    print(f"{split}: {n - len(todo)} cached, {len(todo)} to OCR")

    def one(i):
        ex = dset[i]
        views = [ocr_words(ex["image"], **v, extra_langs=extra_langs) for v in VIEWS]
        return i, {"views": views, "gid": f"sroie_{split}_{i}",
                   "total": to_amount(ex["entities"]["total"]), "date": to_date(ex["entities"]["date"])}

    for s in tqdm(range(0, len(todo), 40), desc=split):
        with ThreadPoolExecutor(WORKERS) as pool:
            for i, d in pool.map(one, todo[s:s + 40]):
                cache[i] = d
        pickle.dump(cache, open(path, "wb"))
    return [cache[i] for i in range(n)]

train_docs = ocr_docs(train_ds, N_TRAIN, "train")
test_docs  = ocr_docs(test_ds,  N_TEST,  "test")
print("OCR done:", len(train_docs), "train,", len(test_docs), "test")

train: 0 cached, 626 to OCR


train:   0%|          | 0/16 [00:00<?, ?it/s]

test: 0 cached, 150 to OCR


test:   0%|          | 0/4 [00:00<?, ?it/s]

OCR done: 626 train, 150 test


## Cell 6 – (Optional, recommended) Your own labelled Sri Lankan bills
Zip your bill images together with a **labels.csv** (columns `file,total,date`, e.g. `bill_001.jpg,1250.00,2024-03-15`) and name it `bills.zip`. 80% are used for training, 20% held out for testing. Skip this cell if you have none yet.

## Cell 7 – Training helpers (tables, models, cross-validation, evaluation)

In [7]:
from sklearn.base import clone
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              HistGradientBoostingClassifier, VotingClassifier)

def make_table(docs):
    parts = []
    for d in docs:
        if not d["total"]:
            continue
        for j, (words, height) in enumerate(d["views"]):
            df = candidates(words, height)
            if df.empty:
                continue
            df["label"] = ((df["value"] - d["total"]).abs() < 0.005).astype(int)
            df["doc"], df["view"], df["true_total"] = d["gid"], j, d["total"]
            parts.append(df)
    return pd.concat(parts, ignore_index=True)

def doc_pick_accuracy(tab, p):
    """Per receipt: sum each value's best probability over the OCR views, pick the top value, compare to truth."""
    t = tab.assign(p=p)
    g = t.groupby(["doc", "view", "value"])["p"].max().groupby(["doc", "value"]).sum().reset_index()
    pick = g.loc[g.groupby("doc")["p"].idxmax()]
    truth = tab.groupby("doc")["true_total"].first().loc[pick["doc"].values].values
    return float((np.abs(pick["value"].values - truth) < 0.005).mean())

def make_models():
    rf = RandomForestClassifier(n_estimators=300, max_depth=14, min_samples_leaf=2,
                                class_weight="balanced", random_state=42, n_jobs=-1)
    et = ExtraTreesClassifier(n_estimators=300, max_depth=14, min_samples_leaf=2,
                              class_weight="balanced", random_state=42, n_jobs=-1)
    hgb = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.08, max_iter=200,
                                         class_weight="balanced", random_state=42)
    return {"RandomForest": rf, "ExtraTrees": et, "HistGB": hgb,
            "Voting(RF+HGB)": VotingClassifier([("rf", clone(rf)), ("hgb", clone(hgb))], voting="soft")}

def cv_compare(tab, k=5):
    models = make_models()
    res = {}
    for name, m in models.items():
        accs = []
        for tr, te in GroupKFold(n_splits=k).split(tab, groups=tab["doc"]):
            mm = clone(m).fit(tab.iloc[tr][FEATURES], tab.iloc[tr]["label"])
            accs.append(doc_pick_accuracy(tab.iloc[te], mm.predict_proba(tab.iloc[te][FEATURES])[:, 1]))
        res[name] = (float(np.mean(accs)), float(np.std(accs)))
        print(f"{name:16s} CV total accuracy: {res[name][0]*100:5.1f}%  (+/- {res[name][1]*100:.1f})")
    return res, models

def evaluate(docs, clf, name, show_errors=10):
    n = ok_t = ok_d = ceil = ocr_miss = 0
    fails, dfails = [], []
    for d in docs:
        if not d["total"]:
            continue
        n += 1
        out = predict_views(d["views"], clf)
        agg = total_scores(d["views"], clf)
        present = any(abs(v - d["total"]) < 0.005 for v in agg)
        ceil += present
        if out["total"] is not None and abs(out["total"] - d["total"]) < 0.005:
            ok_t += 1
        else:
            ocr_miss += (not present)
            fails.append((d.get("gid"), d["total"], out["total"], present,
                          sorted(agg.items(), key=lambda kv: -kv[1])[:3]))
        if out["date"] == d["date"]:
            ok_d += 1
        else:
            dfails.append((d.get("gid"), d["date"], out["date"]))
    print(f"== {name}: {n} receipts")
    print(f"Total correct : {ok_t/n*100:.1f}%")
    print(f"Date correct  : {ok_d/n*100:.1f}%")
    print(f"OCR ceiling   : {ceil/n*100:.1f}%  (total appeared among OCR numbers)")
    print(f"Total errors  : {len(fails)}  ->  {ocr_miss} OCR misses, {len(fails)-ocr_miss} ranking misses")
    for f in fails[:show_errors]:
        print("  total miss:", f)
    for f in dfails[:show_errors]:
        print("  date miss :", f)
    return {"total": ok_t / n, "date": ok_d / n, "ceiling": ceil / n}

## Cell 8 – Compare models with grouped cross-validation, then train the best
Cross-validation never splits a receipt across train and test, and scores the thing you care about: *is the top-ranked number on each receipt the true total?*

In [10]:
OWN_WEIGHT = 3          # your own bills are repeated so SROIE does not drown them out

train_all = train_docs
tab = make_table(train_all)
print(f"{len(tab)} candidate rows from {tab['doc'].nunique()} receipts, {int(tab['label'].sum())} are totals")

cv_res, models = cv_compare(tab, k=5)
best_name = max(cv_res, key=lambda k: cv_res[k][0])
print("\nBest model:", best_name)

clf = clone(models[best_name]).fit(tab[FEATURES], tab["label"])
print("Final model trained on all training receipts.")

11255 candidate rows from 618 receipts, 2207 are totals
RandomForest     CV total accuracy:  79.8%  (+/- 4.2)
ExtraTrees       CV total accuracy:  79.6%  (+/- 2.9)
HistGB           CV total accuracy:  80.9%  (+/- 2.6)
Voting(RF+HGB)   CV total accuracy:  81.4%  (+/- 3.7)

Best model: Voting(RF+HGB)
Final model trained on all training receipts.


## Cell 9 – Test accuracy + error analysis

In [11]:
r1 = evaluate(test_docs, clf, "SROIE test (English)")


== SROIE test (English): 150 receipts
Total correct : 84.7%
Date correct  : 84.7%
OCR ceiling   : 88.0%  (total appeared among OCR numbers)
Total errors  : 23  ->  18 OCR misses, 5 ranking misses
  total miss: ('sroie_test_3', 4.9, 6.0, True, [(6.0, 1.3098520105801155), (4.9, 0.5966740920363205), (5.0, 0.3253645766252502)])
  total miss: ('sroie_test_10', 169.8, 129.7, False, [(129.7, 0.698059868665055), (167.4, 0.4350630043704437), (200.06, 0.14881892214883446)])
  total miss: ('sroie_test_17', 1.05, 1.06, False, [(1.06, 1.5451443178748632), (1.0, 1.5431704181778931), (6.0, 0.10123697784769478)])
  total miss: ('sroie_test_30', 21.6, 30.0, False, [(30.0, 0.4954154759520192), (205.9, 0.13402244903472624), (8.4, 0.06712547766017084)])
  total miss: ('sroie_test_36', 0.9, 5.0, True, [(5.0, 0.5844450963915645), (0.9, 0.453307194651622), (1.0, 0.18213194055434798)])
  total miss: ('sroie_test_41', 5.05, 5.6, False, [(5.6, 0.31924398822958694), (5.04, 0.2558267542977785), (10.0, 0.077420331

**How to read the error analysis**
- *OCR misses* → digits were not read. Fix with better OCR settings (Cell 4) or a stronger OCR engine.
- *Ranking misses* → the total was read but another number was picked. Fix with more training data, especially your own bills.

## Cell 10 – Save and download bill_model.joblib

In [12]:
import joblib, sklearn
from google.colab import files

joblib.dump({"model": clf, "features": FEATURES, "views": VIEWS, "version": 3,
             "model_name": best_name, "sklearn": sklearn.__version__}, "bill_model.joblib")
files.download("bill_model.joblib")
print("scikit-learn version:", sklearn.__version__)
print("IMPORTANT: your app must use the same preprocessing, VIEWS and feature code (Cell 3) as this notebook.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

scikit-learn version: 1.6.1
IMPORTANT: your app must use the same preprocessing, VIEWS and feature code (Cell 3) as this notebook.


## Cell 11 – Try your own bills (no labels needed)

In [14]:
from google.colab import files
uploaded = files.upload()
rows = []
for name in uploaded:
    img = Image.open(name)
    views = [ocr_words(img, **v, extra_langs=("sin", "tam")) for v in VIEWS]
    rows.append({"file": name, **predict_views(views, clf)})
display(pd.DataFrame(rows))

Saving WhatsApp Image 2026-10-22 at 15.27.04.jpeg to WhatsApp Image 2026-10-22 at 15.27.04.jpeg
Saving WhatsApp Image 2026-10-22 at 15.28.17.jpeg to WhatsApp Image 2026-10-22 at 15.28.17.jpeg


,file,total,date,currency
0,WhatsApp Image 2026-10-22 at 15.27.04.jpeg,4749.65,None,LKR
1,WhatsApp Image 2026-10-22 at 15.28.17.jpeg,5030.00,2026-08-16,LKR


In [15]:
from google.colab import files
from IPython.display import display, Javascript

# Save the current modified notebook
display(Javascript('google.colab.notebook.saveNotebook("save")'))

<IPython.core.display.Javascript object>

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

for root, dirs, files_list in os.walk('/content/drive/MyDrive'):
    dirs[:] = [d for d in dirs if d not in ['.trash']]
    for name in files_list:
        if name.endswith('.ipynb'):
            print(os.path.join(root, name))

/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/Copy of Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled3.ipynb
/content/drive/MyDrive/Colab Notebooks/1D_Convolution.ipynb
/content/drive/MyDrive/Colab Notebooks/Image_Filtering_(Convolution) (1).ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled4.ipynb
/content/drive/MyDrive/Colab Notebooks/Image_Filtering_(Convolution).ipynb
/content/drive/MyDrive/Colab Notebooks/CNNs_with_Keras.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled5.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled6.ipynb
/content/drive/MyDrive/Colab Notebooks/Copy of Diabetics.ipynb
/content/drive/MyDrive/Colab Notebooks/Diabetics.ipynb
/content/drive/MyDrive/Colab Notebooks/Heart.ipynb
/content/drive/MyDrive/Colab Notebooks/Parkinson.ipynb
/content/drive/MyDrive/Colab Notebooks/breastcancer.i